In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F

class GroupedQueryAttention(nn.Module):
    """
    Grouped-Query Attention (GQA) as used in Llama 3

    Key insight: Multiple Q heads share the same K,V heads
    Llama 3 70B: 64 Q heads, 8 KV heads (8:1 ratio)

    Benefits:
 - 8x smaller KV cache
 - Faster inference (less memory bandwidth)
 - <0.5% quality loss vs full MHA
    """
    def __init__(
        self,
        hidden_dim: int = 8192,
        n_heads: int = 64,      # Number of Q heads
        n_kv_heads: int = 8,    # Number of KV heads (groups)
        head_dim: int = 128,
    ):
        super().__init__()
        self.n_heads = n_heads
        self.n_kv_heads = n_kv_heads
        self.head_dim = head_dim
        self.n_rep = n_heads // n_kv_heads  # 64 // 8 = 8 Q heads per KV group

        # Q has full n_heads, K,V have n_kv_heads
        self.wq = nn.Linear(hidden_dim, n_heads * head_dim, bias=False)
        self.wk = nn.Linear(hidden_dim, n_kv_heads * head_dim, bias=False)
        self.wv = nn.Linear(hidden_dim, n_kv_heads * head_dim, bias=False)
        self.wo = nn.Linear(n_heads * head_dim, hidden_dim, bias=False)

    def forward(self, x, mask=None):
        batch, seq_len, _ = x.shape

        # Project to Q, K, V
        q = self.wq(x)  # [batch, seq, n_heads * head_dim]
        k = self.wk(x)  # [batch, seq, n_kv_heads * head_dim]  <- Smaller!
        v = self.wv(x)  # [batch, seq, n_kv_heads * head_dim]  <- Smaller!

        # Reshape for attention
        q = q.view(batch, seq_len, self.n_heads, self.head_dim)
        k = k.view(batch, seq_len, self.n_kv_heads, self.head_dim)
        v = v.view(batch, seq_len, self.n_kv_heads, self.head_dim)

        # Repeat K,V to match Q heads (the GQA magic)
        # Each KV head is broadcast to n_rep Q heads
        k = k.repeat_interleave(self.n_rep, dim=2)  # [batch, seq, n_heads, head_dim]
        v = v.repeat_interleave(self.n_rep, dim=2)  # [batch, seq, n_heads, head_dim]

        # Standard attention from here
        q = q.transpose(1, 2)  # [batch, n_heads, seq, head_dim]
        k = k.transpose(1, 2)
        v = v.transpose(1, 2)

        # Scaled dot-product attention
        scale = self.head_dim ** -0.5
        attn = torch.matmul(q, k.transpose(-2, -1)) * scale

        if mask is not None:
            attn = attn.masked_fill(mask == 0, float('-inf'))

        attn = F.softmax(attn, dim=-1)
        out = torch.matmul(attn, v)

        # Reshape and project
        out = out.transpose(1, 2).contiguous().view(batch, seq_len, -1)
        return self.wo(out)

# Example: Llama 3 70B configuration
gqa = GroupedQueryAttention(
    hidden_dim=8192,
    n_heads=64,      # 64 Q heads
    n_kv_heads=8,    # 8 KV heads (8:1 ratio)
    head_dim=128
)

x = torch.randn(1, 1024, 8192)
out = gqa(x)

# KV cache comparison
print("=" * 50)
print("KV CACHE MEMORY COMPARISON (per token, FP16)")
print("=" * 50)
mha_kv = 2 * 64 * 128 * 2  # 2 (K+V) * 64 heads * 128 dim * 2 bytes
gqa_kv = 2 * 8 * 128 * 2   # 2 (K+V) * 8 heads * 128 dim * 2 bytes
print(f"MHA (64 KV heads): {mha_kv:,} bytes per token")
print(f"GQA (8 KV heads):  {gqa_kv:,} bytes per token")
print(f"Reduction: {mha_kv/gqa_kv:.1f}x smaller!")

# For 128K context:
print(f"\nFor 128K context:")
print(f"MHA: {mha_kv * 128000 / 1e9:.1f} GB per layer")
print(f"GQA: {gqa_kv * 128000 / 1e9:.1f} GB per layer")